In [205]:
import pandas as pd

In [206]:
stop_times = pd.read_csv("gtfs_subway/stop_times.txt")
trips = pd.read_csv("gtfs_subway/trips.txt")

In [207]:
stop_times.head(5)

,trip_id,stop_id,arrival_time,departure_time,stop_sequence
0,ASP25GEN-1038-Sunday-00_000600_1..S03R,101S,00:06:00,00:06:00,1
1,ASP25GEN-1038-Sunday-00_000600_1..S03R,103S,00:07:30,00:07:30,2
2,ASP25GEN-1038-Sunday-00_000600_1..S03R,104S,00:09:00,00:09:00,3
3,ASP25GEN-1038-Sunday-00_000600_1..S03R,106S,00:10:30,00:10:30,4
4,ASP25GEN-1038-Sunday-00_000600_1..S03R,107S,00:12:00,00:12:00,5


In [208]:
trips.head()

,route_id,trip_id,service_id,trip_headsign,direction_id,shape_id
0,1,ASP25GEN-1038-Sunday-00_000600_1..S03R,Sunday,South Ferry,1,1..S03R
1,1,ASP25GEN-1038-Sunday-00_002600_1..S03R,Sunday,South Ferry,1,1..S03R
2,1,ASP25GEN-1038-Sunday-00_004600_1..S03R,Sunday,South Ferry,1,1..S03R
3,1,ASP25GEN-1038-Sunday-00_006600_1..S03R,Sunday,South Ferry,1,1..S03R
4,1,ASP25GEN-1038-Sunday-00_007200_1..N03R,Sunday,Van Cortlandt Park-242 St,0,1..N03R


In [209]:
def time_string_to_seconds(time_string):
    hours, minutes, seconds = time_string.split(":")
    return int(hours) * 3600 + int(minutes) * 60 + int(seconds)

In [186]:
def get_route_id(trip_id):
    return trip_id.split("..")[0].split("_")[-1]

In [234]:
def get_short_trip_id(trip_id):
    return "_".join(trip_id.split("-")[-1].split("_")[1:])

In [231]:
HOUR_5PM_SECONDS = time_string_to_seconds("17:00:00")
HOUR_7PM_SECONDS = time_string_to_seconds("19:00:00")

In [189]:
route_ids_of_interest = list("ABCDEFGJLMNQRWZ1234567")

In [192]:
def simplify_route_id(route_id):
    if route_id in ["GS", "SI", "FS"]:
        return "S"
    elif len(route_id) > 1 and route_id[1] == "X":
        return route_id[0]
    return route_id

In [264]:
def get_stop_times():
    stop_times = pd.read_csv("gtfs_subway/stop_times.txt")
    trips = pd.read_csv("gtfs_subway/trips.txt")
    # Convert arrival time to number
    stop_times["arrival_time_s"] = stop_times["arrival_time"].apply(time_string_to_seconds)
    # Filter by arrival time
    stop_times = stop_times[(stop_times["arrival_time_s"] >= HOUR_5PM_SECONDS) & (stop_times["arrival_time_s"] < HOUR_7PM_SECONDS)]
    # Infer route id
    stop_times["inferred_route_id"] = stop_times["trip_id"].apply(get_route_id)
    # Get primary key
    stop_times["trip_id_stop_id"] = stop_times.apply(lambda row: f"{row['trip_id']}_{row['stop_id']}", axis=1)
    # Join with trips
    stop_times = stop_times.merge(
        trips[['trip_id', 'route_id', 'service_id']].rename(columns={'route_id': 'actual_route_id'}), 
        on='trip_id', 
        how='left'
    ).groupby('trip_id_stop_id', as_index=False).first()
    # Get route id with default
    stop_times["route_id"] = stop_times.apply(lambda row: simplify_route_id(row["actual_route_id"]) if "actual_route_id" in row else row["inferred_route_id"], axis=1)
    # Filter out unwanted route ids
    stop_times = stop_times[stop_times["route_id"].apply(lambda r: r in route_ids_of_interest)]
    # Filter out non-weekdays
    stop_times = stop_times[stop_times["service_id"].apply(lambda s: s == "Weekday")]
    # Get short route ids
    stop_times["short_trip_id"] = stop_times["trip_id"].apply(get_short_trip_id)
    return stop_times

In [266]:
stop_times = get_stop_times()
stop_times.head()

,trip_id_stop_id,trip_id,stop_id,arrival_time,departure_time,stop_sequence,arrival_time_s,inferred_route_id,actual_route_id,service_id,route_id,short_trip_id
2857,ASP25GEN-1093-Weekday-00_096150_1..N03R_101N,ASP25GEN-1093-Weekday-00_096150_1..N03R,101N,17:00:00,17:00:00,38,61200,1,1,Weekday,1,096150_1..N03R
2858,ASP25GEN-1093-Weekday-00_096650_1..N03R_101N,ASP25GEN-1093-Weekday-00_096650_1..N03R,101N,17:04:00,17:04:00,38,61440,1,1,Weekday,1,096650_1..N03R
2859,ASP25GEN-1093-Weekday-00_096650_1..N03R_103N,ASP25GEN-1093-Weekday-00_096650_1..N03R,103N,17:01:00,17:02:00,37,61260,1,1,Weekday,1,096650_1..N03R
2860,ASP25GEN-1093-Weekday-00_096700_1..S11R_139S,ASP25GEN-1093-Weekday-00_096700_1..S11R,139S,17:00:00,17:00:00,36,61200,1,1,Weekday,1,096700_1..S11R
2861,ASP25GEN-1093-Weekday-00_096700_1..S11R_142S,ASP25GEN-1093-Weekday-00_096700_1..S11R,142S,17:02:00,17:02:00,37,61320,1,1,Weekday,1,096700_1..S11R


In [268]:
def get_num_stops_with_same_time(df, route_id, arrival_time_s):
    return df[(df["route_id"] != route_id) & (df["arrival_time_s"] == arrival_time_s)]["route_id"].value_counts()

In [269]:
def get_route_unique_stop_times(df, route_id):
    return df[df["route_id"] == route_id]["arrival_time_s"].unique().tolist()

In [270]:
def print_compatability_matrix(df):
    # Initialize the matrix
    result = pd.DataFrame(0, index=route_ids_of_interest, columns=route_ids_of_interest, dtype=int)
    # For each route A
    for route_A in route_ids_of_interest:
        # Get all unique stop times for route A
        stop_times_A = get_route_unique_stop_times(df, route_A)
        # For each stop time in route A
        for stop_time in stop_times_A:
            # Get counts of stops at this time for all other routes
            counts = get_num_stops_with_same_time(df, route_A, stop_time)
            # Update the matrix
            for route_B, count in counts.items():
                if route_B in route_ids_of_interest:
                    result.loc[route_A, route_B] += count
    with pd.option_context('display.max_columns', None):
        print(result)

In [271]:
print_compatability_matrix(stop_times)

      A     B     C     D     E     F    G    J     L     M    N     Q     R  \
A     0  1024  1010  1051  1044  2073  626  750  1589  1098  871  1044  1520   
B  1417     0  1006  1048  1038  2065  624  745  1580  1093  868  1039  1513   
C  1398  1002     0  1027  1031  2033  615  741  1564  1081  858  1025  1505   
D  1417  1019  1006     0  1039  2067  622  745  1581  1092  867  1039  1509   
E  1408  1014  1000  1041     0  2058  621  741  1573  1087  860  1035  1510   
F  1423  1024  1010  1051  1044     0  626  750  1589  1098  871  1044  1520   
G  1367   970   966  1012   978  1992    0  707  1509  1047  834   995  1440   
J  1370   988   969  1017   994  2002  605    0  1526  1044  840  1006  1460   
L  1423  1024  1010  1051  1044  2073  626  750     0  1098  871  1044  1520   
M  1423  1024  1010  1051  1044  2073  626  750  1589     0  871  1044  1520   
N  1420  1016  1004  1047  1041  2062  624  748  1583  1094    0  1039  1514   
Q  1418  1020  1009  1047  1040  2064  6

In [272]:
def get_all_unique_stop_times(df):
    return df["arrival_time_s"].unique().tolist()

In [273]:
def get_all_unique_trains_with_scheduled_time(df, arrival_time_s):
    return df[df["arrival_time_s"] == arrival_time_s].groupby('short_trip_id', as_index=False).first()

In [274]:
def get_all_unique_routes_with_scheduled_time(df, arrival_time_s):
    trains_with_scheduled_time = get_all_unique_trains_with_scheduled_time(df, arrival_time_s)
    return trains_with_scheduled_time["route_id"].unique().tolist()

In [275]:
def map_times_to_num_unique_routes(df):
    unique_stop_times = get_all_unique_stop_times(df)
    stop_time_to_num_unique_routes = []
    for stop_time in unique_stop_times:
        stop_time_to_num_unique_routes.append({
            "arrival_time_s": stop_time,
            "num_routes": len(get_all_unique_lines_with_scheduled_time(stop_times, stop_time))
        })
    return pd.DataFrame(stop_time_to_num_unique_routes)

In [276]:
stop_time_to_num_unique_routes = map_times_to_num_unique_routes(stop_times)

In [277]:
# 240 because scheduled stops are at 30 second increments, 120 minutes * 2 stops / minute = 240 stops
len(stop_time_to_num_unique_routes)

240

In [278]:
stop_time_to_num_unique_routes

,arrival_time_s,num_routes
0,61200,21
1,61440,21
2,61260,21
3,61320,21
4,61350,21
...,...,...
235,68010,21
236,68310,21
237,68370,21
238,68190,20


In [279]:
stop_time_to_num_unique_routes["num_routes"].value_counts()

num_routes
21    127
22     77
20     34
19      2
Name: count, dtype: int64